In [1]:
import gc
import json
import os, h5py
import math
import multiprocessing
import numpy as np
import pandas as pd
import torch
import importlib
import logging
from pathlib import Path
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.utils import resample
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
mp.set_start_method('spawn', force=True)

# Pycox and PyTorch tuples for survival analysis
import torchtuples as tt
import pycox
from pycox.preprocessing.label_transforms import LabTransDiscreteTime
from pycox.models import CoxPH, DeepHit
from pycox.evaluation import EvalSurv

# Ray for hyperparameter tuning and distributed processing
import ray
from ray import tune
from ray.tune import CLIReporter
from ray.tune.search.bayesopt import BayesOptSearch
from ray.tune.search.optuna import OptunaSearch
from ray.tune.search import ConcurrencyLimiter
from ray.tune.schedulers import ASHAScheduler, PopulationBasedTraining
from ray.air import session
import ray.cloudpickle as pickle

# Custom modules for data handling, balancing, training, evaluation, and model architectures
import dataloader2
import databalancer2
import datatrainer2
import modeleval
import netweaver2

# Reload custom modules to ensure latest changes are available
importlib.reload(dataloader2)
importlib.reload(databalancer2)
importlib.reload(datatrainer2)
importlib.reload(modeleval)
importlib.reload(netweaver2)

# Import specific functions from custom modules to keep code clean and readable
from netweaver2 import (
    lstm_net_init, DHANNWrapper, LSTMWrapper, generalized_ann_net_init
)
from dataloader2 import (
    load_and_transform_data, preprocess_data #stack_sequences, dh_dataset_loader
)
from databalancer2 import (
    define_medoid_general, df_event_focus, rebalance_data, underbalance_data_general, medoid_cluster, 
    dh_rebalance_data
)
from datatrainer2 import (
    recursive_clustering, prepare_training_data, 
    prepare_validation_data, lstm_training
)
from modeleval import (
    dh_test_model, nam_dagostino_chi2, get_baseline_hazard_at_timepoints, combined_test_model
)

import psutil
torch.cuda.empty_cache()
gc.collect()

239

In [2]:
# Define Constants and Load Datasets
RANDOM_SEED = 12345
N_SPLIT = 2
FEATURE_COLS = ['gender', 'dm', 'ht', 'sprint', 'a1c', 'po4', 'UACR_mg_g', 'Cr', 'age', 'alb', 'ca', 'hb', 'hco3']
DURATION_COL = 'date_from_sub_60'
EVENT_COL = 'endpoint'
A_CLASS_COL = 'A_class'
G_CLASS_COL = 'G_class'
CLUSTER_COL = 'key'
TIME_GRID = np.array([i * 365 for i in range(6)])

# Define Feature Groups
CAT_FEATURES = ['gender', 'dm', 'ht', 'sprint']
LOG_FEATURES = ['a1c', 'po4', 'UACR_mg_g', 'Cr']
STANDARD_FEATURES = ['age', 'alb', 'ca', 'hb', 'hco3']
PASSTHROUGH_FEATURES = ['key', 'date_from_sub_60', 'endpoint']

# Load and Transform Data
BASE_FILENAME = '/mnt/d/pydatascience/g3_regress/data/X/X_20240628'
X_train_transformed, X_test_transformed = load_and_transform_data(
    BASE_FILENAME, CAT_FEATURES, LOG_FEATURES, STANDARD_FEATURES, PASSTHROUGH_FEATURES
)

2025-04-27 08:13:43,479 - INFO - Transforming training data...
2025-04-27 08:13:58,085 - INFO - Transforming test data...
/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [3]:
def create_neural_network(config, num_risk = len(X_train_transformed[EVENT_COL].unique()) - 1, num_time_bins=len(TIME_GRID)):
    """
    Function to create a neural network based on the given configuration.

    Args:
        config (dict): Configuration dictionary containing model type, network type, and hyperparameters.

    Returns:
        torch.nn.Module: Created neural network model.
    """
    gc.collect()
    torch.cuda.empty_cache()
    if config['model'] == 'deepsurv':
        num_risk = None
        num_time_bins=None
    elif config['model'] == 'deephit':
        num_risk = num_risk
        num_time_bins = num_time_bins
    # Create the Neural Network
    if config['net'] == 'ann':
        net = generalized_ann_net_init(
            input_size=len(config['features']),
            num_nodes=config["num_nodes"],
            batch_norm=config["batch_norm"],
            dropout=config["dropout"],
            output_size=1, # Default output size for DeepSurv
            num_risks = num_risk,
            num_time_bins = num_time_bins
        )
    elif config['net'] == 'lstm':
        net = lstm_net_init(
            input_size=len(config['features']),
            num_nodes=config["num_nodes"],
            batch_norm=config["batch_norm"],
            dropout=config["dropout"],
            num_risks = num_risk,
            num_time_bins = num_time_bins
        )
    else:
        raise ValueError("Unknown network type: {}".format(config['net']))

    optimizer = tt.optim.AdamWR(decoupled_weight_decay=1e-6, cycle_eta_multiplier=0.8)
    if config['model'] == 'deepsurv':
        model = CoxPH(net, optimizer)
    elif config['model'] == 'deephit':
        model = DeepHit(net, optimizer)
    model.optimizer.set_lr(config["lr"])
    
    return model

def train_neural_network(model, config, X_train, X_val, duration_col, event_col, a_class_col, g_class_col, cluster_col, callbacks, time_grid=None):
    """
    Function to train a given neural network using the provided datasets.

    Args:
        net (torch.nn.Module): Neural network to be trained.
        config (dict): Configuration dictionary containing model hyperparameters.
        X_train (pd.DataFrame): Training dataset with features.
        X_val (pd.DataFrame): Validation dataset with features.
        duration_col (str): Column representing event durations.
        event_col (str): Column representing event occurrences.
        cluster_col (str): Column for grouping during cross-validation.
        callbacks (list): List of callbacks for training.
        time_grid (np.array, optional): Time grid for evaluation if required. Defaults to None.

    Returns:
        model: Trained PyCox model.
        logs: Training logs.
    """
    gc.collect()
    torch.cuda.empty_cache()
    # Train the model
    if config['model'] == 'deepsurv':
        print('Initiate training of deepsurv neural network')
        X_val = df_event_focus(X_val, event_col, config['endpoint'])
        X_val_processed, y_val = preprocess_data(X_val, config['features'], duration_col, event_col, a_class_col, g_class_col)
        y_val = (y_val[0], y_val[1])
        val_data = (X_val_processed, y_val)
        if config['net'] == 'ann':
            print('model structure: ANN')
            if config['balance_method'] == 'clustering':
                print('data balancing method: clustering')
                model, logs = recursive_clustering(model, X_train, duration_col, event_col, a_class_col, g_class_col, config, val_data, callbacks, max_repeats=30)
            elif config['balance_method'] == 'enn':
                print('data balancing method: smoteenn')
                X_train = X_train.drop(columns=[a_class_col, g_class_col])
                X_train = rebalance_data(X_train, event_col, config['endpoint'], CAT_FEATURES, config, RANDOM_SEED, method='ENN')
                X_train, y_train = preprocess_data(X_train, config['features'], duration_col, event_col, a_class_col, g_class_col)
                logs = model.fit(X_train, y_train, config['batch_size'], int(config['max_epochs']), callbacks, verbose=True, val_data=val_data, num_workers=10)
            elif config['balance_method'] == 'tomek':
                print('data balancing method: smotetomek')
                X_train = X_train.drop(columns=[a_class_col, g_class_col])
                X_train = rebalance_data(X_train, event_col, config['endpoint'], CAT_FEATURES, config, RANDOM_SEED, method='Tomek')
                X_train, y_train = preprocess_data(X_train, config['features'], duration_col, event_col, a_class_col, g_class_col)
                y_train = (y_train[0], y_train[1])
                logs = model.fit(X_train, y_train, config['batch_size'], int(config['max_epochs']), callbacks, verbose=True, val_data=val_data, num_workers=10)
        elif config['net'] == 'lstm':
            print('model structure: LSTM')
            if config['balance_method'] == 'clustering':
                print('data balancing method: clustering')
                X_train = X_train.drop(columns=[a_class_col, g_class_col])
                X_val = X_val.drop(columns=[a_class_col, g_class_col])
                model, logs = lstm_training(model, X_train, X_val, duration_col, event_col, cluster_col, config, callbacks, time_grid)
            elif config['balance_method'] == 'NearMiss':
                print('data balancing method: NearMiss')
                X_train = X_train.drop(columns=[a_class_col, g_class_col])
                X_val = X_val.drop(columns=[a_class_col, g_class_col])
                model, logs = lstm_training(model, X_train, X_val, duration_col, event_col, cluster_col, config, callbacks, time_grid)
    elif config['model'] == 'deephit':
        print('Initiate training of deephit neural network')
        X_val_processed, y_val = preprocess_data(X_val, config['features'], duration_col, event_col, a_class_col, g_class_col, TIME_GRID, discretize=True)
        y_val = (y_val[0], y_val[1])
        val_data = (X_val_processed, y_val)
        if config['net'] == 'ann':
            print('model structure: ANN')
            if config['balance_method'] == 'clustering':
                print('data balancing method: clustering')
                X_train = X_train.drop(columns=[a_class_col, g_class_col])
                model, logs = recursive_clustering(model, X_train, duration_col, event_col, a_class_col, g_class_col, config, val_data, callbacks, max_repeats=30, time_grid=TIME_GRID)
            elif config['balance_method'] == 'NearMiss':
                print('data balancing method: NearMiss')
                X_train = X_train.drop(columns=[a_class_col, g_class_col])
                X_train = underbalance_data_general(X_train, event_col, cluster_col, config, version=config['version'])
                X_train, y_train = preprocess_data(X_train, config['features'], duration_col, event_col, a_class_col, g_class_col, TIME_GRID, discretize=True)
                y_train = (y_train[0], y_train[1])
                logs = model.fit(X_train, y_train, config['batch_size'], int(config['max_epochs']), callbacks, verbose=True, val_data=val_data)
        elif config['net'] == 'lstm':
            print('model structure: LSTM')
            if config['balance_method'] == 'clustering':
                print('data balancing method: clustering')
                X_train = X_train.drop(columns=[a_class_col, g_class_col])
                X_val = X_val.drop(columns=[a_class_col, g_class_col])
                model, logs = lstm_training(model, X_train, X_val, duration_col, event_col, cluster_col, config, callbacks, time_grid)
            elif config['balance_method'] == 'NearMiss':
                print('data balancing method: NearMiss')
                X_train = X_train.drop(columns=[a_class_col, g_class_col])
                X_val = X_val.drop(columns=[a_class_col, g_class_col])
                model, logs = lstm_training(model, X_train, X_val, duration_col, event_col, cluster_col, config, callbacks, time_grid)        

    # Free memory after training
    gc.collect()
    torch.cuda.empty_cache()

    return model, logs

def save_model(params, model, model_path, baseline_hazard_path):
    """
    Save model weights and baseline hazard data.

    Parameters:
    - model: The trained model to save.
    - model_path: Path to save the model weights (.pt file).
    - baseline_hazard_path: Path to save the baseline hazards (.pkl file).
    """
    # Compute baseline hazards and save
    if params['model'] == 'deepsurv':
        baseline_hazard = model.compute_baseline_hazards()
        baseline_hazard.to_pickle(baseline_hazard_path)
    
    # Save model weights
    model.save_model_weights(model_path)
    print(f"Model and baseline hazards saved to {model_path} and {baseline_hazard_path}.")

def training_wrapper(df, config, spliter, model_path, hazard_path, feature_col=FEATURE_COLS, duration_col=DURATION_COL, event_col=EVENT_COL, 
                     a_class_col=A_CLASS_COL, g_class_col=G_CLASS_COL, 
                     cluster_col=CLUSTER_COL, time_grid=TIME_GRID):
    """
    Train and save a survival analysis model with grouped cross-validation splits.

    This function performs training on grouped cross-validation splits of the input DataFrame and saves each trained model
    along with its baseline hazards. Memory management is handled to ensure efficient GPU usage.

    Parameters:
    - df (pd.DataFrame): DataFrame containing training data.
    - config (dict): Configuration dictionary for initializing the neural network.
    - spliter (object): Splitter object (e.g., GroupShuffleSplit or StratifiedKFold) used for creating train-validation splits.
    - model_path (str): File path to save the trained model weights (.pt file).
    - hazard_path (str): File path to save the baseline hazards (.pkl file).
    - feature_col (list): List of feature column names in `df` used for model training.
    - duration_col (str): Name of the column representing duration/time-to-event.
    - event_col (str): Name of the column representing the event indicator (0 = censored, 1 = event).
    - cluster_col (str): Name of the column used for grouping (clusters for cross-validation).
    - time_grid (list): List or array defining the time grid for training.

    Returns:
    - None: Saves the model weights and baseline hazard data for each cross-validation split.
    """
    for train_idx, val_idx in spliter.split(X=df[feature_col], y=df[event_col], groups=df[cluster_col]):
        # Clear GPU memory for each split
        gc.collect()
        torch.cuda.empty_cache()
        
        # Define early stopping callback
        callbacks = [tt.cb.EarlyStopping()]
        
        # Create training and validation sets
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        # Initialize and train the model
        model = create_neural_network(config)
        model, logs = train_neural_network(
            model, config,
            X_train=train_df, X_val=val_df,
            duration_col=duration_col, event_col=event_col,
            a_class_col=a_class_col, g_class_col=g_class_col,
            cluster_col=cluster_col, callbacks=callbacks, time_grid=time_grid
        )
        
        # Save the trained model and its baseline hazards
        save_model(config, model, model_path, hazard_path)
        
        # Free memory for the next iteration
        del model, logs
        gc.collect()
        torch.cuda.empty_cache()

    print("Training and saving completed for all cross-validation splits.")

    print("All models have been trained and saved successfully.")

In [4]:
model_ls = ['deepsurv_ann_clustering_1', 'deepsurv_ann_smoteenn_1', 'deepsurv_ann_smotetomek_1',
            'deepsurv_ann_clustering_2', 'deepsurv_ann_smoteenn_2', 'deepsurv_ann_smotetomek_2',
            'deepsurv_lstm_clustering_1', 'deepsurv_lstm_nearmiss_1', 'deepsurv_lstm_clustering_2', 'deepsurv_lstm_nearmiss_2',
            'deephit_ann_clustering_all', 'deephit_ann_nearmiss2_all', 'deephit_lstm_clustering_all', 'deephit_lstm_nearmiss1_all']
config_path = '/mnt/d/pydatascience/g3_regress/code/models/20250423/all_model_configs.json'
model_path = '/mnt/d/pydatascience/g3_regress/code/models/20250423/'
# Load the JSON file
with open(config_path, "r") as json_file:
    model_configs = json.load(json_file)

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
for train_idx_1, fin_val_idx in gss1.split(X=X_train_transformed[FEATURE_COLS], y=X_train_transformed[EVENT_COL], groups=X_train_transformed[CLUSTER_COL]):
    X_train_transformed_2, X_fin_val = X_train_transformed.iloc[train_idx_1, :], X_train_transformed.iloc[fin_val_idx, :]
    gc.collect()
    torch.cuda.empty_cache()
    for model in model_configs.keys():
        model_config = model_configs[model]
        if model_config is None:
            print(f"Configuration for {model} not found.")
            continue

        model_weights_path = f'{model_path}{model}.pt'
        model_hazard_path = f'{model_path}{model}_hazard.pkl'
        
        training_wrapper(X_train_transformed_2, model_config, gss2, model_weights_path, 
                        model_hazard_path, 
                        feature_col=FEATURE_COLS, duration_col=DURATION_COL, event_col=EVENT_COL, 
                        a_class_col=A_CLASS_COL, g_class_col=G_CLASS_COL, cluster_col=CLUSTER_COL, time_grid=TIME_GRID)
        gc.collect()
        torch.cuda.empty_cache()

2025-04-27 08:13:59,684 - INFO - Event column 'endpoint' updated with focus on event value 1.
2025-04-27 08:13:59,713 - INFO - Event column 'endpoint' updated with focus on event value 1.
2025-04-27 08:13:59,753 - INFO - Performing clustering iteration 1 / 20
2025-04-27 08:13:59,754 - INFO - init
2025-04-27 08:13:59,756 - INFO - CUDA environment set up and GPU memory cleared.
2025-04-27 08:13:59,780 - INFO - Event column 'endpoint' updated with focus on event value 1.


Initiate training of deepsurv neural network
model structure: ANN
data balancing method: clustering


2025-04-27 08:14:00,539 - INFO - Defined medoid for deepsurv model with 1207 clusters.
/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/torchtuples/callbacks.py:607: UserWarning: This overload of add is deprecated:
	add(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add(Tensor other, *, Number alpha = 1) (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:1581.)
  p.data = p.data.add(-weight_decay * eta, p.data)


0:	[0s / 0s],		train_loss: 5.0672,	val_loss: 7.7351
1:	[0s / 0s],		train_loss: 4.9376,	val_loss: 7.2237
2:	[0s / 0s],		train_loss: 4.8863,	val_loss: 7.2362
3:	[0s / 0s],		train_loss: 4.8350,	val_loss: 6.9229
4:	[0s / 0s],		train_loss: 4.8161,	val_loss: 6.8952
5:	[0s / 0s],		train_loss: 4.8125,	val_loss: 6.9862
6:	[0s / 0s],		train_loss: 4.8308,	val_loss: 6.9099
7:	[0s / 0s],		train_loss: 4.8033,	val_loss: 6.9808
8:	[0s / 0s],		train_loss: 4.8068,	val_loss: 7.0187


/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/torchtuples/base.py:669: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.net.load_state_dict(torch.load(pat

...

64249


/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/torch/nn/modules/rnn.py:917: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at ../aten/src/ATen/native/cudnn/RNN.cpp:1424.)
  result = _VF.lstm(input, hx, self._flat_weights, self.bias, self.num_layers,


0:	[9s / 9s],		train_loss: 0.4838,	val_loss: 0.0904
1:	[9s / 19s],		train_loss: 0.4270,	val_loss: 0.0735
2:	[12s / 32s],		train_loss: 0.4006,	val_loss: 0.0749
3:	[9s / 41s],		train_loss: 0.3880,	val_loss: 0.0745
4:	[9s / 50s],		train_loss: 0.3619,	val_loss: 0.0756
5:	[12s / 1m:3s],		train_loss: 0.3384,	val_loss: 0.0634
6:	[9s / 1m:12s],		train_loss: 0.3218,	val_loss: 0.0586
7:	[9s / 1m:22s],		train_loss: 0.3497,	val_loss: 0.0488
8:	[12s / 1m:35s],		train_loss: 0.3278,	val_loss: 0.0522
Model and baseline hazards saved to /mnt/d/pydatascience/g3_regress/code/models/20250423/deephit_lstm_nearmiss1_all.pt and /mnt/d/pydatascience/g3_regress/code/models/20250423/deephit_lstm_nearmiss1_all_hazard.pkl.


/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/torchtuples/base.py:669: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.net.load_state_dict(torch.load(pat

Training and saving completed for all cross-validation splits.
All models have been trained and saved successfully.


In [5]:
def load_model(model, model_config, model_path, baseline_hazard_path):
    """
    Load model weights and baseline hazard data.

    Parameters:
    - create_model_func: Function to create the model architecture (e.g., create_neural_network).
    - model_path: Path to load the model weights (.pt file).
    - baseline_hazard_path: Path to load the baseline hazards (.pkl file).

    Returns:
    - model: The loaded model with weights and baseline hazards.
    """
    
    # Load model weights
    model.load_model_weights(model_path)
    
    # Load baseline hazards and assign to model
    if model_config['model'] == 'deepsurv':
        baseline_hazard = pd.read_pickle(baseline_hazard_path)
        model.baseline_hazards_ = baseline_hazard
        model.baseline_cumulative_hazards_ = baseline_hazard.cumsum()
    
    print(f"Model and baseline hazards loaded from {model_path} and {baseline_hazard_path}.")
    return model

In [ ]:
def predict_neural_network(model, config, X_test, duration_col, event_col, a_class_col, g_class_col, cluster_col, time_grid=None):
    """
    Function to train a given neural network using the provided datasets.

    Args:
        net (torch.nn.Module): Neural network to be trained.
        config (dict): Configuration dictionary containing model hyperparameters.
        X_train (pd.DataFrame): Training dataset with features.
        X_val (pd.DataFrame): Validation dataset with features.
        duration_col (str): Column representing event durations.
        event_col (str): Column representing event occurrences.
        cluster_col (str): Column for grouping during cross-validation.
        callbacks (list): List of callbacks for training.
        time_grid (np.array, optional): Time grid for evaluation if required. Defaults to None.

    Returns:
        model: Trained PyCox model.
        logs: Training logs.
    """
    gc.collect()
    torch.cuda.empty_cache()
    # Train the model
    if config['model'] == 'deepsurv':
        print('Initiate testing of deepsurv neural network')
        X_test = df_event_focus(X_test, event_col, config['endpoint'])
        if config['net'] == 'ann':
            print('model structure: ANN')
            X_test_processed, y_test = preprocess_data(X_test, config['features'], duration_col, event_col, a_class_col, g_class_col)
            # y_test = (y_test[0], y_test[1])
            surv = model.predict_surv_df(X_test_processed, batch_size=512)
        elif config['net'] == 'lstm':
            print('model structure: LSTM')
            print(f"{config['model']}, {config['net']}, {config['balance_method']}, {config['endpoint']}")
            X_test_processed, y_test = prepare_validation_data(X_test, config['features'], duration_col, event_col, config, cluster_col, config['model'], time_grid)
            X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
            durations_test = np.asarray(y_test[0], dtype=np.float32)
            events_test = np.asarray(y_test[1], dtype=np.int64)
            y_test_tensor  = (torch.from_numpy(durations_test),
                                torch.from_numpy(events_test))
            surv = model.predict_surv_df(X_test_tensor, batch_size=512)
    elif config['model'] == 'deephit':
        print('Initiate testing of deephit neural network')
        if config['net'] == 'ann':
            print('model structure: ANN')
            X_test_processed, y_test = preprocess_data(X_test, config['features'], duration_col, event_col, a_class_col, g_class_col, time_grid, discretize=True)
            # y_test = (y_test[0], y_test[1])
            surv = model.predict_cif(X_test_processed, batch_size=512)
            print('prediction complete, please note that prediction of deephit models are CIF.')
        elif config['net'] == 'lstm':
            print('model structure: LSTM')
            X_test_processed, y_test = prepare_validation_data(X_test, config['features'], duration_col, event_col, config, cluster_col, config['model'], time_grid)
            surv = model.predict_cif(X_test_processed, batch_size=512)
            print('prediction complete, please note that prediction of deephit models are CIF.')

    # Free memory after training
    gc.collect()
    torch.cuda.empty_cache()

    return surv, y_test

def align_to_time_grid(surv, time_grid):
    """
    Align the survival DataFrame to the closest indices of the time grid.

    Parameters:
        surv (pd.DataFrame): Survival probabilities DataFrame.
        time_grid (np.array): Array of target time points to align.

    Returns:
        aligned_surv (pd.DataFrame): Aligned survival probabilities.
    """
    # Convert the DataFrame's index to a NumPy array for fast computation
    surv_times = np.array(surv.index)
    
    # Find the closest time in the survival DataFrame for each time in the grid
    closest_indices = [np.argmin(np.abs(surv_times - t)) for t in time_grid]
    
    # Extract the rows corresponding to the closest times
    aligned_surv = surv.iloc[closest_indices].copy()
    
    # Reindex the DataFrame to match the time grid
    aligned_surv.index = range(len(time_grid))  # Standardize indices to 0, 1, 2, ...
    return aligned_surv

In [7]:
cif_array_labels = [
    "deepsurv_ann_clustering",
    "deepsurv_ann_enn",
    "deepsurv_ann_tomek",
    "deepsurv_lstm_clustering",
    "deepsurv_lstm_NearMiss",
    "deephit_ann_clustering",
    "deephit_ann_NearMiss",
    "deephit_lstm_clustering",
    "deephit_lstm_NearMiss",
]

def prediction_wrapper(df, feature_col, duration_col, event_col, a_class_col, g_class_col, cluster_col, time_grid, config_path, model_path, cif_array_labels):
    
    
    """
    Perform and output predictions, ground truth durations, and ground truth events.

    Args:
        df (DataFrame): Test dataset with a 'key' column.
        feature_cols (list): List of feature columns.
        duration_col (str): Column name for duration.
        event_col (str): Column name for event types.
        cluster_col (str): Column name for clustering (e.g., patient identifier).
        time_grid (list): Time points for evaluation.
        config_path (str): path storing each model configs.
        model_path (str): path storing model weights and hazards.
        cif_array_labels (list): list of model labels to locate cif predictions in output.
    """
    
    gc.collect()
    torch.cuda.empty_cache()
    # Step 1: load the models needed
    with open(config_path, "r") as json_file:
        model_configs = json.load(json_file)
    loaded_models = {}

    for model_name in model_configs.keys():
        model_config = model_configs[model_name]
        if model_config is None:
            print(f"Configuration for {model_name} not found.")
            continue
        model_weights_path = f'{model_path}{model_name}.pt'
        model_hazard_path = f'{model_path}{model_name}_hazard.pkl'
        create_model_func = lambda: create_neural_network(
            config=model_config,
            num_risk=len(df[event_col].unique()) - 1,
            num_time_bins=len(time_grid)
        )
        model = create_model_func()
        loaded_models[model_name] = load_model(model, model_config, model_weights_path, model_hazard_path)
        
    # Step 2: Prepare ground truth from df 
    _, y = preprocess_data(df, feature_col, duration_col, event_col, a_class_col, g_class_col, time_grid, discretize=True)
    
    # Step 3: Make prediction
    predict_cif_dict = {model: np.zeros((len(df[event_col].unique())-1, len(time_grid), df.shape[0])) for model in cif_array_labels}
    for model_name in loaded_models.keys():
        model = loaded_models[model_name]
        model_config = model_configs[model_name]
        surv, _ = predict_neural_network(
                model, model_config,
                df,
                duration_col, event_col, a_class_col, g_class_col, 
                cluster_col,
                time_grid
            )
        # Align survival probabilities (if DeepSurv)
        if model_config['model'] == 'deepsurv':
            surv = align_to_time_grid(surv, TIME_GRID).values  # 2D array
                
            # Structure key dynamically
            key = f"deepsurv_{model_config['net']}_{model_config['balance_method']}"
            assert surv.shape == predict_cif_dict[key][model_config['endpoint']-1].shape
            predict_cif_dict[key][model_config['endpoint']-1] = 1 - surv
            
        # Handle DeepHit predictions
        elif model_config['model'] == 'deephit':
            surv = np.array(surv)  # Convert to numpy array
                
            # Structure key dynamically
            key = f"deephit_{model_config['net']}_{model_config['balance_method']}"
            assert surv.shape == predict_cif_dict[key].shape
            predict_cif_dict[key] = surv
    for key in predict_cif_dict.keys():
        assert predict_cif_dict[key].shape[-1] == y[0].shape[0] == y[1].shape[0]
    return predict_cif_dict, y

In [8]:
import h5py
import os
from sklearn.utils import resample
import gc
import torch


def single_bootstrap_iteration(
    iteration_idx,
    df,
    feature_col,
    duration_col,
    event_col,
    a_class_col, 
    g_class_col,
    cluster_col,
    time_grid,
    config_path,
    model_path,
    cif_array_labels,
    output_dir
):
    """
    Perform a single bootstrap iteration and save the result to a separate HDF5 file.

    Args:
        iteration_idx (int): The index of the bootstrap iteration.
        df (DataFrame): The input dataframe.
        feature_col (list): Feature columns.
        duration_col (str): Duration column.
        event_col (str): Event column.
        cluster_col (str): Cluster column.
        time_grid (list): Time grid.
        config_path (str): Path to model configurations.
        model_path (str): Path to model weights.
        cif_array_labels (list): List of CIF array labels.
        output_dir (str): Directory to save HDF5 results.
    """
    print(f"Bootstrap Iteration {iteration_idx + 1}")

    # Resample keys with replacement
    unique_keys = df[cluster_col].unique()
    resampled_keys = resample(unique_keys, replace=True)

    # Filter the data by resampled keys
    resampled_data = df[df[cluster_col].isin(resampled_keys)]
    print(f"Total rows in resampled data for iteration {iteration_idx + 1}: {len(resampled_data)}")

    # Perform predictions
    predictions, truth = prediction_wrapper(
        resampled_data,
        feature_col,
        duration_col,
        event_col,
        a_class_col, 
        g_class_col,
        cluster_col,
        time_grid,
        config_path,
        model_path,
        cif_array_labels
    )
    # Define unique file name for this iteration
    output_file = os.path.join(output_dir, f"bootstrap_iteration_{iteration_idx + 1}.h5")

    # Save results to a separate HDF5 file
    with h5py.File(output_file, "w") as hdf:
        # Save predictions
        pred_group = hdf.create_group("predictions")
        for key, value in predictions.items():
            pred_group.create_dataset(key, data=value)
        
        # Save durations and events
        hdf.create_dataset("durations", data=truth[0])
        hdf.create_dataset("events", data=truth[1])
        hdf.create_dataset("a_class", data=np.char.encode(truth[2], encoding='utf-8'))
        hdf.create_dataset("g_class", data=np.char.encode(truth[3], encoding='utf-8'))

    print(f"Saved bootstrap iteration {iteration_idx + 1} to {output_file}.")


def bootstrap_predictions(
    df,
    feature_col,
    duration_col,
    event_col,
    a_class_col, 
    g_class_col,
    cluster_col,
    time_grid,
    config_path,
    model_path,
    cif_array_labels,
    n_bootstrap,
    output_dir
):
    """
    Perform bootstrap iterations sequentially and save each result to a separate HDF5 file.

    Args:
        df (DataFrame): Input dataframe containing test data.
        feature_col (list): List of feature columns.
        duration_col (str): Column name for duration.
        event_col (str): Column name for events.
        cluster_col (str): Column name for patient clustering.
        time_grid (list): Time grid for predictions.
        config_path (str): Path to model configurations in JSON format.
        model_path (str): Path to directory containing model weights and hazards.
        cif_array_labels (list): List of CIF array labels (model names).
        n_bootstrap (int): Number of bootstrap iterations.
        output_dir (str): Directory to save the HDF5 files.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    for i in range(n_bootstrap):
        try:
            single_bootstrap_iteration(
                i,
                df,
                feature_col,
                duration_col,
                event_col,
                a_class_col, 
                g_class_col,
                cluster_col,
                time_grid,
                config_path,
                model_path,
                cif_array_labels,
                output_dir
            )
        except Exception as e:
            print(f"Error in bootstrap iteration {i + 1}: {e}")

    print(f"Bootstrap completed. Results saved to {output_dir}.")
    gc.collect()
    torch.cuda.empty_cache()


In [9]:
X_train_transformed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 396424 entries, 0 to 396423
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   gender            396424 non-null  category
 1   dm                396424 non-null  category
 2   ht                396424 non-null  category
 3   sprint            396424 non-null  category
 4   a1c               396424 non-null  float64 
 5   po4               396424 non-null  float64 
 6   UACR_mg_g         396424 non-null  float64 
 7   Cr                396424 non-null  float64 
 8   age               396424 non-null  float64 
 9   alb               396424 non-null  float64 
 10  ca                396424 non-null  float64 
 11  hb                396424 non-null  float64 
 12  hco3              396424 non-null  float64 
 13  key               396424 non-null  int64   
 14  date_from_sub_60  396424 non-null  object  
 15  endpoint          396424 non-null  object  
 16  eG

In [12]:
def chunk_and_save_df(
    df: pd.DataFrame,
    chunk_size: int,
    output_dir: str,
    prefix: str = "train_chunk",
    file_format: str = "h5"
):
    os.makedirs(output_dir, exist_ok=True)
    total = len(df)
    n_chunks = (total + chunk_size - 1) // chunk_size

    for idx in range(n_chunks):
        start = idx * chunk_size
        end = min(start + chunk_size, total)
        chunk_df = df.iloc[start:end]

        filename = f"{prefix}_{idx+1:03d}.{file_format}"
        filepath = os.path.join(output_dir, filename)

        if file_format == "h5":
            # <-- add format="table" to allow categorical dtypes
            chunk_df.to_hdf(filepath, key="df", mode="w", format="table")
        elif file_format == "parquet":
            chunk_df.to_parquet(filepath)
        elif file_format == "pkl":
            chunk_df.to_pickle(filepath)
        else:
            raise ValueError(f"unsupported format {file_format!r}")

        yield filepath


In [24]:
X_train_transformed['date_from_sub_60'] = X_train_transformed['date_from_sub_60'].astype(int)

# 2. now sort by key then by that integer date
X_train_transformed = (
    X_train_transformed
    .sort_values(['key', 'date_from_sub_60'], ascending=[True, True])
    .reset_index(drop=True)
)

all_keys = X_train_transformed['key'].unique()
key_chunks = np.array_split(all_keys, np.ceil(len(all_keys) / 300).astype(int))


In [28]:
X_train_transformed.loc[X_train_transformed['key'].isin(key_chunks[0])]

,gender,dm,ht,sprint,a1c,po4,UACR_mg_g,Cr,age,alb,ca,hb,hco3,key,date_from_sub_60,endpoint,eGFRcr,A_class,G_class
0,1.0,1.0,1.0,0.0,0.317849,0.569661,0.802511,0.360691,0.688889,0.526316,0.360189,0.487923,0.361377,449,5,0.0,47.754788,A3,G3a
1,1.0,1.0,1.0,0.0,0.317849,0.569661,0.840609,0.344638,0.688889,0.438597,0.360189,0.492754,0.422563,449,7,0.0,52.529108,A3,G3a
2,1.0,1.0,1.0,0.0,0.352882,0.569661,0.844112,0.382607,0.700000,0.438597,0.360189,0.492754,0.422563,449,175,0.0,41.669476,A3,G3b
3,1.0,1.0,1.0,0.0,0.280205,0.569661,0.842581,0.381218,0.700000,0.438597,0.360189,0.492754,0.422563,449,287,0.0,42.014564,A3,G3b
4,1.0,1.0,1.0,0.0,0.280205,0.569661,0.842710,0.382607,0.700000,0.438597,0.360189,0.492754,0.422563,449,287,0.0,41.669476,A3,G3b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3674,1.0,0.0,1.0,0.0,0.131627,0.596224,0.750008,0.363754,0.788889,0.543860,0.339239,0.400967,0.401530,55190,1126,0.0,44.341608,A2,G3b
3675,1.0,0.0,1.0,0.0,0.131627,0.633123,0.750008,0.372674,0.788889,0.491229,0.337561,0.386474,0.382410,55190,1269,0.0,42.054807,A2,G3b
3676,1.0,0.0,1.0,0.0,0.131627,0.608492,0.750008,0.347952,0.800000,0.438597,0.378697,0.371981,0.382410,55190,1547,0.0,48.400101,A2,G3a
3677,1.0,0.0,1.0,0.0,0.135210,0.616340,0.750008,0.375563,0.800000,0.438597,0.378697,0.371981,0.382410,55190,1632,0.0,41.083692,A2,G3b


In [29]:
config_path = '/mnt/d/pydatascience/g3_regress/code/models/20250423/all_model_configs.json'
output_base = "/mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_test/"

for idx, keys in enumerate(key_chunks, start=1):
    chunk_name = f"keys_{idx:03d}"
    out_dir = os.path.join(output_base, chunk_name)
    os.makedirs(out_dir, exist_ok=True)

    # 3. slice
    df_slice = X_train_transformed.loc[X_train_transformed['key'].isin(keys)]
    
    print(f"\n=== Running bootstrap for chunk {idx:03d} ({len(keys)} keys, {len(df_slice)} rows) ===")

    # 4. feed to your function
    bootstrap_predictions(
        df=df_slice,
        feature_col=FEATURE_COLS,
        duration_col=DURATION_COL,
        event_col=EVENT_COL,
        a_class_col=A_CLASS_COL,
        g_class_col=G_CLASS_COL,
        cluster_col=CLUSTER_COL,
        time_grid=TIME_GRID,
        config_path=config_path,
        model_path="/mnt/d/pydatascience/g3_regress/code/models/20250423/",
        cif_array_labels=cif_array_labels,
        n_bootstrap=1,
        output_dir=out_dir
    )

print("All chunks processed.")


=== Running bootstrap for chunk 001 (298 keys, 3679 rows) ===
Bootstrap Iteration 1
Total rows in resampled data for iteration 1: 2123


/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/torchtuples/base.py:669: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.net.load_state_dict(torch.load(pat

Model and baseline hazards loaded from /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_clustering_1.pt and /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_clustering_1_hazard.pkl.
Model and baseline hazards loaded from /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_smoteenn_1.pt and /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_smoteenn_1_hazard.pkl.


/home/goma/miniconda3/envs/ai_dev/lib/python3.11/site-packages/torchtuples/base.py:669: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.net.load_state_dict(torch.load(pat

Model and baseline hazards loaded from /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_smotetomek_1.pt and /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_smotetomek_1_hazard.pkl.
Model and baseline hazards loaded from /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_clustering_2.pt and /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_clustering_2_hazard.pkl.
Model and baseline hazards loaded from /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_smoteenn_2.pt and /mnt/d/pydatascience/g3_regress/code/models/20250423/deepsurv_ann_smoteenn_2_hazard.pkl.


...

marker 0
marker 3
Initiate testing of deephit neural network
model structure: ANN
prediction complete, please note that prediction of deephit models are CIF.
Initiate testing of deephit neural network
model structure: ANN
prediction complete, please note that prediction of deephit models are CIF.
Initiate testing of deephit neural network
model structure: LSTM


2025-04-27 17:28:15,000 - INFO - Validation data retrieved


prediction complete, please note that prediction of deephit models are CIF.
Initiate testing of deephit neural network
model structure: LSTM


2025-04-27 17:28:16,033 - INFO - Validation data retrieved


prediction complete, please note that prediction of deephit models are CIF.
Saved bootstrap iteration 1 to /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_test/keys_099/bootstrap_iteration_1.h5.
Bootstrap completed. Results saved to /mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_test/keys_099.
All chunks processed.
All chunks processed.


In [ ]:
config_path = '/mnt/d/pydatascience/g3_regress/code/models/20250423/all_model_configs.json'
output_path = "/mnt/d/pydatascience/g3_regress/data/results/bootstrap_predictions/20250427_test/"
bootstrap_predictions(
    df=X_test_transformed,
    feature_col=FEATURE_COLS,
    duration_col=DURATION_COL,
    event_col=EVENT_COL,
    a_class_col=A_CLASS_COL, 
    g_class_col=G_CLASS_COL,
    cluster_col=CLUSTER_COL,
    time_grid=TIME_GRID,
    config_path=config_path,
    model_path="/mnt/d/pydatascience/g3_regress/code/models/20250423/",
    cif_array_labels=cif_array_labels,
    n_bootstrap=500,
    output_dir=output_path
)
